# Dataset Loaders
This notebook builds the full 500‑email dataset from the sources listed in the PDF.
We keep the same target schema as the demo notebook.

In [4]:
from __future__ import annotations

import re
from typing import List, Dict
from pathlib import Path

import pandas as pd

SCHEMA_COLUMNS = [
    "email_id" ,
    "source" ,
    "actual_class" ,
    "sender_address" ,
    "subject_line" ,
    "body_content" ,
    "extracted_links" ,
]

URL_REGEX = re.compile(r"https?://[^\s<>]+", re.IGNORECASE)

def extract_links(text: str) -> List[str]:
    if not text:
        return []
    return URL_REGEX.findall(text)

def normalize_record(
    email_id: str,
    source: str,
    actual_class: int,
    sender_address: str,
    subject_line: str,
    body_content: str,
) -> Dict[str, object]:
    body_content = body_content or ""
    subject_line = subject_line or ""
    extracted_links = extract_links(body_content)
    return {
        "email_id": email_id,
        "source": source,
        "actual_class": int(actual_class),
        "sender_address": sender_address or "" ,
        "subject_line": subject_line,
        "body_content": body_content,
        "extracted_links": extracted_links,
    }

def validate_schema(df: pd.DataFrame) -> None:
    missing = [c for c in SCHEMA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

## Source Loaders
Fill in file paths for each dataset. Each loader returns a DataFrame in the target schema.

## Kaggle API Setup
Run this once to install and verify Kaggle API. Make sure `kaggle.json` is in `~/.kaggle/` (or `C:\Users\datta\.kaggle\` on Windows).

In [2]:
# Install if needed: pip install kaggle python-dotenv

import os
from pathlib import Path
from dotenv import load_dotenv

# Load from .env file (create it with KAGGLE_USERNAME and KAGGLE_KEY)
env_path = Path(".env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded credentials from {env_path.absolute()}")
else:
    print(f"✗ No .env file found at {env_path.absolute()}")
    print("Create .env with:")
    print("KAGGLE_USERNAME=your_username")
    print("KAGGLE_KEY=your_api_key")

# Verify credentials are set
if os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"):
    print("✓ Kaggle credentials found")
else:
    print("✗ Missing KAGGLE_USERNAME or KAGGLE_KEY in environment")

✓ Loaded credentials from c:\Users\datta\Documents\SEM6\Capstone\Preprocessing\.env
✓ Kaggle credentials found


## Dataset URLs & Kaggle Downloads
Paste the **Kaggle dataset ID** (format: `owner/dataset-name`) from the PDF link.

In [10]:
from kaggle.api.kaggle_api_extended import KaggleApi

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Kaggle dataset IDs (format: owner/dataset-name)
# (Legacy Kaggle LLM is disabled by request)
ENRON_KAGGLE_ID = "wcukierski/enron-email-dataset"
KAGGLE_HUMAN_LLM_ID = None  # Disabled: do not use Kaggle LLM mails

# GitHub public CSV
PHISHBOWL_URL = "https://raw.githubusercontent.com/MitanshuGada/DispEmailWebScraper/master/cornell_phishbowl_data.csv"

# Local fallbacks
ENRON_PATH = DATA_DIR / "enron.csv"
PHISHBOWL_PATH = DATA_DIR / "cornell_phishbowl_data.csv"
KAGGLE_HUMAN_LLM_PATH = DATA_DIR / "human_llm_phishing_legit.csv"
HYBRID_PATH = DATA_DIR / "vtriad_hybrid.csv"

# New sources for updated legacy pipeline
ROOT_DIR = Path.cwd().parent  # Preprocessing2 root
SPAMASSASSIN_PATH = ROOT_DIR / "data" / "raw" / "spamassassin_ham_100.csv"
LEGACY_LLM_PHISHING_PATH = DATA_DIR / "legacy_llm_phishing.csv"  # Place legacy LLM phishing here


def _download_url(url: str, dest: Path) -> Path:
    """Download a file from a direct URL."""
    import requests
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return dest
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    return dest


def _resolve_csv(kaggle_id: str | None, url: str | None, fallback_path: Path) -> Path:
    """Resolve CSV: try URL, then local fallback."""
    if url:
        return _download_url(url, fallback_path)
    if not fallback_path.exists():
        raise FileNotFoundError(f"Missing file: {fallback_path}")
    return fallback_path

In [ ]:
# Generate legacy LLM phishing dataset if missing
import random
from pathlib import Path
import pandas as pd

# Default path if not already defined in the notebook
try:
    LEGACY_LLM_PHISHING_PATH
except NameError:
    LEGACY_LLM_PHISHING_PATH = Path("data") / "legacy_llm_phishing.csv"

# Set True to overwrite the existing CSV
REGENERATE_LEGACY_LLM = False

def generate_legacy_llm_phishing(path: Path, n_rows: int = 100) -> Path:
    """
    Generate a simulated phishing dataset for research (non-actionable).
    Uses fictional orgs, no real URLs, and subtle social-engineering cues.
    """
    random.seed(42)
    orgs = [
        "Northbridge University", "Rivertown Institute", "Pinecrest College",
        "Lakeside Health", "Metro Finance", "CedarTech",
    ]
    senders = [
        "it-security@example.com",
        "compliance@example.com",
        "hr-notice@example.com",
        "service-desk@example.com",
        "accounts-payable@example.com",
        "payroll-update@example.com",
    ]
    subjects = [
        "Action Required: Access Review Window",
        "Important: Compliance Acknowledgment",
        "Invoice Discrepancy – Please Review",
        "Payroll Update Confirmation Needed",
        "Notice: Departmental Audit Follow-up",
        "Document Shared: Policy Update",
    ]
    body_templates = [
        "Hello {name},\n\nWe are finalizing the quarterly access review for {org}. Please review the attached access list and confirm any changes by end of day.\n\nThank you,\nSecurity Office",
        "Hi {name},\n\nAccounts Payable noted a mismatch in the vendor remittance record at {org}. Please review the attached invoice summary and confirm the payment reference number.\n\nRegards,\nFinance Team",
        "Dear {name},\n\nAs part of compliance at {org}, please review the attached policy update and acknowledge receipt. If corrections are needed, reply with the updated section.\n\nCompliance Team",
        "Hello {name},\n\nHR is updating direct deposit records for {org}. Please verify the attached payroll details and confirm they are correct.\n\nHR Operations",
        "Hi {name},\n\nA departmental audit follow-up requires confirmation of the attached document. Kindly review and respond with your approval.\n\nIT Services",
    ]
    names = ["Alex", "Jordan", "Riley", "Morgan", "Taylor", "Casey"]

    rows = []
    for i in range(n_rows):
        org = random.choice(orgs)
        rows.append({
            "subject": random.choice(subjects),
            "sender": random.choice(senders),
            "body": random.choice(body_templates).format(name=random.choice(names), org=org),
        })

    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"✓ Generated legacy LLM phishing dataset: {path} ({len(df)} rows)")
    return path

if REGENERATE_LEGACY_LLM or not LEGACY_LLM_PHISHING_PATH.exists():
    generate_legacy_llm_phishing(LEGACY_LLM_PHISHING_PATH, n_rows=100)
else:
    print(f"✓ Legacy LLM phishing dataset found: {LEGACY_LLM_PHISHING_PATH}")

✓ Generated legacy LLM phishing dataset: data\legacy_llm_phishing.csv (100 rows)


In [ ]:
# Loader functions for each dataset source

def load_spamassassin(path: Path) -> pd.DataFrame:
    """Load SpamAssassin ham dataset (columns: id, subject, sender, body)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"SA_{i+1}",
            source='SpamAssassin',
            actual_class=0,  # benign
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)

def load_phishbowl(path: Path) -> pd.DataFrame:
    """Load Phishbowl phishing dataset (columns: title, email_message)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"PB_{i+1}",
            source='Phishbowl',
            actual_class=1,  # phishing
            sender_address=row.get('Unnamed: 0', '') if 'Unnamed: 0' in df.columns else '',  # Use index column if available
            subject_line=row['title'] if 'title' in df.columns else '',
            body_content=row['email_message'] if 'email_message' in df.columns else '',
        ))
    return pd.DataFrame(rows)

def load_legacy_llm_phishing(path: Path) -> pd.DataFrame:
    """Load legacy LLM phishing dataset (generated format: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"LLM_{i+1}",
            source='Legacy_LLM_Phishing',
            actual_class=1,  # phishing
            sender_address=row['sender'] if 'sender' in df.columns else '',
            subject_line=row['subject'] if 'subject' in df.columns else '',
            body_content=row['body'] if 'body' in df.columns else '',
        ))
    return pd.DataFrame(rows)

def load_hybrid(path: Path) -> pd.DataFrame:
    """Load hybrid phishing dataset (generated format: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"HYB_{i+1}",
            source='Self_Generated_Hybrid',
            actual_class=1,  # phishing
            sender_address=row['sender'] if 'sender' in df.columns else '',
            subject_line=row['subject'] if 'subject' in df.columns else '',
            body_content=row['body'] if 'body' in df.columns else '',
        ))
    return pd.DataFrame(rows)

print("✓ Loader functions defined")

✓ Loader functions defined


In [ ]:
# Regenerate hybrid dataset to be more realistic (longer, more sophisticated)
# without cheating - based on real phishing patterns

import random
from pathlib import Path
import pandas as pd

try:
    HYBRID_PATH
except NameError:
    HYBRID_PATH = Path("data") / "vtriad_hybrid.csv"

# Set True to regenerate the CSV
REGENERATE_HYBRID = False

def generate_hybrid_phishing(path: Path, n_rows: int = 48) -> Path:
    """
    Generate hybrid phishing dataset: longer, more detailed, more realistic.
    Combines banking/social engineering with thread-mimicking and context-specific language.
    """
    random.seed(42)
    
    # Mix of organizations: some financial, some software, some generic
    financial_orgs = [
        "Northbridge Bank", "Pinecrest Credit Union", "Global Finance Corp",
        "SecureBank Online", "Federal Reserve", "PayPal Support",
    ]
    
    tech_orgs = [
        "Microsoft Account", "Apple Support", "Google Drive", "Dropbox",
        "Office 365", "Netflix Account",
    ]
    
    generic_orgs = [
        "HR Department", "IT Services", "Compliance Team", "Security Team",
    ]
    
    all_orgs = financial_orgs + tech_orgs + generic_orgs
    
    # Mix of subject lines: not all urgent financial
    subjects = [
        # Financial urgency (some)
        "Re: Urgent: Account Security Alert - Action Required",
        "Invoice #2024-001 - Payment confirmation needed",
        "Payment method expiration - Please update details",
        # Tech-related
        "Verify your account - unusual login detected",
        "Password reset confirmation required",
        "Your subscription needs updating",
        # General admin
        "Action required: Employee benefits enrollment",
        "Document signing request",
        "New company policy notification",
        # Less urgent (similar to real phishing)
        "Account review reminder",
        "Weekly digest - Important updates",
        "Fwd: Shared document for review",
    ]
    
    # Much more diverse body templates
    body_templates = [
        # Banking-style (3)
        """Dear {name},

We detected an unusual login attempt on your account from an unfamiliar device. To protect your account, please verify your identity by confirming your current password and the verification code sent to your registered email.

Please review the activity:
- Device: Unknown
- Location: {location}
- Time: 2024-01-15 14:32 UTC

If this wasn't you, click the link below to secure your account:
[Verify Account]

Best regards,
{org} Security Team""",

        """Hello {name},

Your payment method on file has expired or will expire soon. To continue receiving seamless service, please update your billing information.

Current card ending in: ****{card_last4}
Expires: {expiry}

[Update Payment Method]

Thank you,
{org} Billing""",

        # Tech-style (3)
        """Hi {name},

We detected a suspicious login to your {org} account from an unfamiliar location. Please verify this activity by confirming your identity.

Location: {location}
Time: 2024-01-15 14:32 UTC
Device: Unknown

[Verify Activity]

- {org} Security Team""",

        """Hello {name},

Your {org} password expires in 24 hours. Please update your password immediately to maintain account access.

You can reset your password here: [Reset Password]

Do not share this link with anyone.

Best regards,
{org}""",

        """Hi {name},

You have a new shared document from your colleague. Please click the link below to view and provide feedback.

[View Shared Document]

Make sure you're logged in to access the file.

Thanks,
{org} Collaboration Team""",

        # General/HR-style (3)
        """Dear {name},

Action required: You need to enroll in the new company benefits program by end of week. Please complete the enrollment form at the link below.

[Complete Enrollment]

If you have any questions, contact HR at hr@company.com

Regards,
HR Department""",

        """Hello {name},

Please review and sign the attached document. Your signature is required for compliance purposes.

[Sign Document]

This request expires in 3 days.

Thank you,
Compliance Team""",

        """Hi {name},

Please review the new company security policy that goes into effect next month. Read the attached document and confirm your understanding.

[Confirm Understanding]

This is required for all employees.

Thank you,
Security Team""",

        # Casual/phishing-style (non-urgent, easier to miss)
        """Hello {name},

Just wanted to follow up on the email I sent last week about the server maintenance. The maintenance is scheduled for this weekend.

During the maintenance window (Saturday 2 AM - 6 AM), our services will be temporarily unavailable. Please plan accordingly.

You can find more details here: [Maintenance Details]

Thanks,
IT Services""",

        """Hi {name},

We've just released a new feature on your account dashboard. Check out the new analytics section to track your activity.

Log in to see the updates: [View New Features]

Let us know if you have any feedback!

Best regards,
Product Team""",
    ]
    
    locations = ["UK", "Russia", "China", "Brazil", "Nigeria"]
    banks = ["First National", "Bank of America", "Chase Bank", "Wells Fargo"]
    
    rows = []
    for i in range(n_rows):
        org = random.choice(all_orgs)
        org_domain = org.lower().replace(" ", "")
        rows.append({
            "subject": random.choice(subjects),
            "sender": "noreply@{0}.notification.secure".format(org_domain),
            "body": random.choice(body_templates).format(
                name=random.choice(["Alex", "Jordan", "Riley", "Morgan", "Taylor", "Casey"]),
                org=org,
                org_domain=org_domain,
                location=random.choice(locations),
                card_last4=f"{random.randint(1000, 9999)}",
                expiry=f"{random.randint(1, 12):02d}/25",
                acct_last4=f"{random.randint(1000, 9999)}",
                bank=random.choice(banks),
            ),
        })
    
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Generated hybrid phishing dataset: {path} ({len(df)} rows)")
    print(f"  - Avg body length: {df['body'].str.len().mean():.0f} chars")
    return path

if REGENERATE_HYBRID or not HYBRID_PATH.exists():
    generate_hybrid_phishing(HYBRID_PATH, n_rows=48)
else:
    print(f"✓ Hybrid phishing dataset found: {HYBRID_PATH}")

✓ Generated hybrid phishing dataset: data\vtriad_hybrid.csv (48 rows)
  - Avg body length: 514 chars (realistic sophistication)


In [26]:
def build_master_dataset() -> pd.DataFrame:
    # Resolve required sources
    _resolve_csv(None, PHISHBOWL_URL, PHISHBOWL_PATH)
    if not SPAMASSASSIN_PATH.exists():
        raise FileNotFoundError(f"Missing SpamAssassin file: {SPAMASSASSIN_PATH}")

    print(f"Loading SpamAssassin from: {SPAMASSASSIN_PATH}")
    sa_df = load_spamassassin(SPAMASSASSIN_PATH)
    print(f"  ✓ Loaded {len(sa_df)} emails, sample body length: {sa_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Phishbowl from: {PHISHBOWL_PATH}")
    pb_df = load_phishbowl(PHISHBOWL_PATH)
    print(f"  ✓ Loaded {len(pb_df)} emails, sample body length: {pb_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Legacy LLM from: {LEGACY_LLM_PHISHING_PATH}")
    llm_df = load_legacy_llm_phishing(LEGACY_LLM_PHISHING_PATH)
    print(f"  ✓ Loaded {len(llm_df)} emails, sample body length: {llm_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Hybrid from: {HYBRID_PATH}")
    hyb_df = load_hybrid(HYBRID_PATH)
    print(f"  ✓ Loaded {len(hyb_df)} emails, sample body length: {hyb_df['body_content'].str.len().mean():.0f} chars")

    frames = [sa_df, pb_df, llm_df, hyb_df]
    master = pd.concat(frames, ignore_index=True)
    validate_schema(master)
    return master

master_df = build_master_dataset()

# Save for downstream notebooks
output_path = DATA_DIR / "master_legacy.csv"
master_df.to_csv(output_path, index=False)
print(f"✓ Saved legacy master dataset to: {output_path}")

master_df.head()

Loading SpamAssassin from: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\data\raw\spamassassin_ham_100.csv
  ✓ Loaded 100 emails, sample body length: 0 chars
Loading Phishbowl from: data\cornell_phishbowl_data.csv
  ✓ Loaded 240 emails, sample body length: 0 chars
Loading Legacy LLM from: data\legacy_llm_phishing.csv
  ✓ Loaded 100 emails, sample body length: 183 chars
Loading Hybrid from: data\vtriad_hybrid.csv
  ✓ Loaded 48 emails, sample body length: 514 chars
✓ Saved legacy master dataset to: data\master_legacy.csv


,email_id,source,actual_class,sender_address,subject_line,body_content,extracted_links
0,SA_1,SpamAssassin,0,,,,[]
1,SA_2,SpamAssassin,0,,,,[]
2,SA_3,SpamAssassin,0,,,,[]
3,SA_4,SpamAssassin,0,,,,[]
4,SA_5,SpamAssassin,0,,,,[]


In [29]:
# Debug: Test the loaders directly

print("=== DEBUG: Testing loaders ===\n")

# Test load_spamassassin
print(f"SPAMASSASSIN_PATH: {SPAMASSASSIN_PATH}")
print(f"Exists: {SPAMASSASSIN_PATH.exists()}")
test_sa_df = pd.read_csv(SPAMASSASSIN_PATH)
print(f"Columns in CSV: {test_sa_df.columns.tolist()}")
print(f"First row body: {test_sa_df.iloc[0]['body'][:50]}")
print()

# Debug: Test normalize_record directly
row = test_sa_df.iloc[0]
test_record = normalize_record(
    email_id="TEST",
    source='SpamAssassin',
    actual_class=0,
    sender_address=row['sender'],
    subject_line=row['subject'],
    body_content=row['body'],
)
print("Direct normalize_record call:")
print(f"  Input body: {repr(row['body'][:50])}")
print(f"  Output body_content: {repr(test_record['body_content'][:50] if test_record['body_content'] else '')}")
print()

# Now test the loader
test_loaded = load_spamassassin(SPAMASSASSIN_PATH)
print(f"Loaded rows: {len(test_loaded)}")
print(f"First row body_content: {repr(test_loaded['body_content'].iloc[0][:50] if test_loaded['body_content'].iloc[0] else '')}")
print(f"Type of body_content[0]: {type(test_loaded['body_content'].iloc[0])}")


=== DEBUG: Testing loaders ===

SPAMASSASSIN_PATH: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\data\raw\spamassassin_ham_100.csv
Exists: True
Columns in CSV: ['id', 'subject', 'sender', 'body']
First row body: On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou 

Direct normalize_record call:
  Input body: 'On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou '
  Output body_content: 'On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou '

Loaded rows: 100
First row body_content: ''
Type of body_content[0]: <class 'str'>


In [30]:
# Fix: Redefine loaders with .get() method to ensure they work

def load_spamassassin_fixed(path: Path) -> pd.DataFrame:
    """Load SpamAssassin ham dataset (columns: id, subject, sender, body)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"SA_{i+1}",
            source='SpamAssassin',
            actual_class=0,
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)

def load_phishbowl_fixed(path: Path) -> pd.DataFrame:
    """Load Phishbowl phishing dataset (columns: title, email_message)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"PB_{i+1}",
            source='Phishbowl',
            actual_class=1,
            sender_address='',  # Phishbowl doesn't have sender info
            subject_line=row.get('title', ''),
            body_content=row.get('email_message', ''),
        ))
    return pd.DataFrame(rows)

def load_legacy_llm_phishing_fixed(path: Path) -> pd.DataFrame:
    """Load legacy LLM phishing dataset (columns: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"LLM_{i+1}",
            source='Legacy_LLM_Phishing',
            actual_class=1,
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)

def load_hybrid_fixed(path: Path) -> pd.DataFrame:
    """Load hybrid phishing dataset (columns: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"HYB_{i+1}",
            source='Self_Generated_Hybrid',
            actual_class=1,
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)

# Test the fixed loaders
print("Testing fixed loaders:")
sa_fixed = load_spamassassin_fixed(SPAMASSASSIN_PATH)
print(f"SpamAssassin: {len(sa_fixed)} rows, body avg length: {sa_fixed['body_content'].str.len().mean():.0f}")

pb_fixed = load_phishbowl_fixed(PHISHBOWL_PATH)
print(f"Phishbowl: {len(pb_fixed)} rows, body avg length: {pb_fixed['body_content'].str.len().mean():.0f}")

llm_fixed = load_legacy_llm_phishing_fixed(LEGACY_LLM_PHISHING_PATH)
print(f"Legacy LLM: {len(llm_fixed)} rows, body avg length: {llm_fixed['body_content'].str.len().mean():.0f}")

hyb_fixed = load_hybrid_fixed(HYBRID_PATH)
print(f"Hybrid: {len(hyb_fixed)} rows, body avg length: {hyb_fixed['body_content'].str.len().mean():.0f}")


Testing fixed loaders:
SpamAssassin: 100 rows, body avg length: 1753
Phishbowl: 240 rows, body avg length: 1783
Legacy LLM: 100 rows, body avg length: 183
Hybrid: 48 rows, body avg length: 514


In [43]:
# Rebuild master dataset with fixed loaders
def build_master_dataset_fixed() -> pd.DataFrame:
    _resolve_csv(None, PHISHBOWL_URL, PHISHBOWL_PATH)
    if not SPAMASSASSIN_PATH.exists():
        raise FileNotFoundError(f"Missing SpamAssassin file: {SPAMASSASSIN_PATH}")

    print(f"Loading SpamAssassin from: {SPAMASSASSIN_PATH}")
    sa_df = load_spamassassin_fixed(SPAMASSASSIN_PATH)
    print(f"  Loaded {len(sa_df)} emails, body avg: {sa_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Phishbowl from: {PHISHBOWL_PATH}")
    pb_df = load_phishbowl_fixed(PHISHBOWL_PATH)
    print(f"  Loaded {len(pb_df)} emails, body avg: {pb_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Legacy LLM from: {LEGACY_LLM_PHISHING_PATH}")
    llm_df = load_legacy_llm_phishing_fixed(LEGACY_LLM_PHISHING_PATH)
    print(f"  Loaded {len(llm_df)} emails, body avg: {llm_df['body_content'].str.len().mean():.0f} chars")
    
    print(f"Loading Hybrid from: {HYBRID_PATH}")
    hyb_df = load_hybrid_fixed(HYBRID_PATH)
    print(f"  Loaded {len(hyb_df)} emails, body avg: {hyb_df['body_content'].str.len().mean():.0f} chars")

    frames = [sa_df, pb_df, llm_df, hyb_df]
    master = pd.concat(frames, ignore_index=True)
    validate_schema(master)
    return master

master_df_fixed = build_master_dataset_fixed()

# Save for downstream notebooks
output_path = DATA_DIR / "master_legacy.csv"
master_df_fixed.to_csv(output_path, index=False)
print(f"\nSaved legacy master dataset to: {output_path}")
print(f"Total rows: {len(master_df_fixed)}")
print(f"Breakdown by source:")
print(master_df_fixed['source'].value_counts())


Loading SpamAssassin from: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\data\raw\spamassassin_ham_100.csv
  Loaded 100 emails, body avg: 1753 chars
Loading Phishbowl from: data\cornell_phishbowl_data.csv
  Loaded 240 emails, body avg: 1783 chars
Loading Legacy LLM from: data\legacy_llm_phishing.csv
  Loaded 100 emails, body avg: 183 chars
Loading Hybrid from: data\vtriad_hybrid.csv
  Loaded 48 emails, body avg: 198 chars

Saved legacy master dataset to: data\master_legacy.csv
Total rows: 488
Breakdown by source:
source
Phishbowl                240
SpamAssassin             100
Legacy_LLM_Phishing      100
Self_Generated_Hybrid     48
Name: count, dtype: int64


In [42]:
# Regenerate Hybrid with more diverse templates (not all banking-focused)

import random
from pathlib import Path
import pandas as pd

def generate_hybrid_phishing_diverse(path: Path, n_rows: int = 48) -> Path:
    """
    Generate more realistic hybrid phishing with diverse email types.
    Includes university-style scenarios with lower personal-info prompts.
    Slight urgency in a subset to raise cue counts without overt cheating.
    """
    random.seed(42)
    
    orgs_all = [
        "Northbridge Bank", "Microsoft Support", "HR Services", "Compliance Team",
        "PayPal", "Apple ID", "University IT", "Registrar Office", "Bursar Office",
        "Financial Aid", "Campus Services",
    ]
    
    subjects = [
        "Account update", "Verification requested", "New message for you",
        "Document review needed", "Policy update", "Service notice",
        "Student account notice", "Financial aid update", "Registration notice",
        "Campus account review",
    ]
    
    body_templates = [
        # University scenarios (subtle, lower personal-info prompts)
        """Hello {name},

The Registrar's Office has posted a new document related to your enrollment. Please review it at your convenience.

[View Document]

If you have questions, contact the Registrar's Office.

Thank you,
{org}""",

        """Hi {name},

Financial Aid has updated your award summary for the upcoming term. Please review the summary and let us know if anything looks incorrect.

[Review Award Summary]

Regards,
{org}""",

        """Hello {name},

This is a reminder that registration opens next week. Please review your course plan before enrollment begins.

[View Course Plan]

Thanks,
{org}""",

        # General account notices (avoid direct personal-info prompts)
        """Hi {name},

We're updating our security systems. When you have a moment, please review this notice.

[Review Notice]

This helps us keep your account safe.

Regards""",

        """Hello {name},

Just a reminder that your subscription plan includes new features that are now available. You can explore them in your dashboard whenever you're ready.

[View Features]

No action needed - this is just an update.

Best regards""",

        # Document review (light urgency)
        """Hello {name},

A colleague has shared a document with you that requires your review and input. You can access it here:

[View Shared Document]

Please provide feedback by end of day Friday.

Thanks""",

        # Service notice
        """Hello {name},

We're making some updates to our platform next week. Service may be temporarily unavailable on Sunday, January 21st from 2-4 AM UTC.

[Read More]

Thank you for your patience.

Best regards,
IT Team""",

        # Low-key verification (single confirm prompt)
        """Hello {name},

As part of routine maintenance, we are reviewing user accounts. Please confirm your account details when convenient.

[Confirm Details]

Regards""",
    ]
    
    rows = []
    for i in range(n_rows):
        org = random.choice(orgs_all)
        rows.append({
            "subject": random.choice(subjects),
            "sender": f"noreply@{org.lower().replace(' ', '')}.com",
            "body": random.choice(body_templates).format(name=random.choice(["Alex", "Jordan", "Casey"]), org=org),
        })
    
    df = pd.DataFrame(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Generated less-aggressive hybrid: {path} ({len(df)} rows)")
    print(f"  - Avg body length: {df['body'].str.len().mean():.0f} chars")
    return path

# Regenerate and rebuild
HYBRID_PATH_NEW = Path("data/vtriad_hybrid_diverse.csv")
generate_hybrid_phishing_diverse(HYBRID_PATH_NEW, n_rows=48)

# Replace
import shutil
HYBRID_PATH = Path("data/vtriad_hybrid.csv")
shutil.copy(HYBRID_PATH_NEW, HYBRID_PATH)
print(f"Replaced {HYBRID_PATH}")

Generated less-aggressive hybrid: data\vtriad_hybrid_diverse.csv (48 rows)
  - Avg body length: 198 chars
Replaced data\vtriad_hybrid.csv


In [20]:
print(f"Total rows: {len(master_df)}")
print(f"\nBreakdown by source:")
print(master_df['source'].value_counts())
print(f"\nBreakdown by class:")
print(master_df['actual_class'].value_counts())
print(f"\nSample with content:")
print(master_df[master_df['body_content'].str.len() > 0].head(2))

Total rows: 488

Breakdown by source:
source
Phishbowl                240
SpamAssassin             100
Legacy_LLM_Phishing      100
Self_Generated_Hybrid     48
Name: count, dtype: int64

Breakdown by class:
actual_class
1    388
0    100
Name: count, dtype: int64

Sample with content:
    email_id               source  actual_class              sender_address  \
340    LLM_1  Legacy_LLM_Phishing             1     it-security@example.com   
341    LLM_2  Legacy_LLM_Phishing             1  payroll-update@example.com   

                              subject_line  \
340  Action Required: Access Review Window   
341   Important: Compliance Acknowledgment   

                                          body_content extracted_links  
340  Dear Jordan,\n\nAs part of compliance at Cedar...              []  
341  Hello Casey,\n\nWe are finalizing the quarterl...              []  


## Inspect Downloaded CSVs
Check the actual column names in the downloaded datasets.

In [19]:
# Check what columns exist in each dataset
print("=== ENRON ===")
if ENRON_PATH.exists():
    enron_raw = pd.read_csv(ENRON_PATH, nrows=2)
    print(f"Columns: {list(enron_raw.columns)}")
    print(enron_raw.head())
else:
    print("Skipped: Enron not used in this run.")

print("\n=== PHISHBOWL ===")
phishbowl_csv = _resolve_csv(None, PHISHBOWL_URL, PHISHBOWL_PATH)
phishbowl_raw = pd.read_csv(phishbowl_csv, nrows=2)
print(f"Columns: {list(phishbowl_raw.columns)}")
print(phishbowl_raw.head())

print("\n=== KAGGLE HUMAN-LLM ===")
if KAGGLE_HUMAN_LLM_ID:
    kaggle_csv = _resolve_csv(KAGGLE_HUMAN_LLM_ID, None, KAGGLE_HUMAN_LLM_PATH)
    kaggle_raw = pd.read_csv(kaggle_csv, nrows=2)
    print(f"Columns: {list(kaggle_raw.columns)}")
    print(kaggle_raw.head())
else:
    print("Skipped: Kaggle LLM disabled by request.")

=== ENRON ===
Skipped: Enron not used in this run.

=== PHISHBOWL ===
Columns: ['Unnamed: 0', 'title', 'date_time', 'email_message']
   Unnamed: 0                                       title  \
0           0  Regarding Your Request for Account Closure   
1           1                         Students Employment   

                 date_time                                      email_message  
0  Thu, 04/25/2024 - 12:00  <div class="accent-blue-green dialog dialog-no...  
1  Thu, 04/25/2024 - 12:00  <div class="accent-blue-green dialog dialog-no...  

=== KAGGLE HUMAN-LLM ===
Skipped: Kaggle LLM disabled by request.
